In [3]:
################### Sensitivity analysis to get optimal VZ threshold ##############

### Threshold definition: VZ threshold is define as that if (1) the final prediction class is VZ and (2) the probability output from RF model
    ### is less than this threshold, then re-classify VZ to TZ
### Justification: some of the VZ predictions have low probability, resulting in large differences of VZ across years.
    ### Applying the VZ threshold smooths the temporal changes in the map time series.
### Method: using the nearest two GE years for model training, and then applying the VZ threshold ########
    ### Last, validate prediction results using the consistent points between two GE years ################
#### ThresholdVZ is selected as 0.6 considering all the results from three Landsat periods
from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyproj import Proj, transform
import ast
from pyproj import Transformer
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling

from scipy.ndimage import binary_fill_holes
from scipy.ndimage import convolve
from scipy.ndimage import distance_transform_edt
from affine import Affine
from scipy.spatial import KDTree
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape
from simplekml import Kml
from osgeo import gdal
from rasterio.transform import rowcol
from sklearn.metrics import accuracy_score


In [4]:
def count_non_nan_values(array):
        """Counts non-NaN values in an array while ignoring non-numeric elements."""
        # Filter to keep only numeric data (e.g., ints and floats)
        numeric_values = [x for x in array if isinstance(x, (int, float))] 
        # Convert to NumPy array and count non-NaN values
        return np.sum(~np.isnan(np.array(numeric_values, dtype=float)))
    
def data_prep(path, file, cutoff=0, direction=0, cutlon=0, sensor = 1):
    ####### filter the data with 015042 index ######
    df = pd.read_csv(path + file)
    df['ID'] = range(0, len(df))  # Generates IDs starting from 1
    
    if sensor == 0:
        training_data = df[['lat', 'lon', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    else:
        training_data = df[['lat', 'lon', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    
    predictor = []
    target = []
    coord = []
    # Loop through each (1585, 10) array in training_data
#     print('training data is:')
#     #print(training_data)
#     print(training_data.shape)
    #for array in training_data:
        # Get rows where the target is not 'TZ'
    mask = training_data[:, 11] != 'MZ'
    
    predictor.append(training_data[mask, 2:11])  # First 9 columns
    target.append(training_data[mask, 11])       # Target column
    coord.append(training_data[mask, :2])
    
    # Convert to NumPy arrays if desired
    predictor = np.array(predictor)
    target = np.array(target)
    coord = np.array(coord)
    
    X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
    y = target.reshape(-1)        # Shape: (5 * 1585,)
    coord = coord.reshape(-1, 2)
    ### cleanse the data by removing nan ####
    valid_indices = []

    # Loop through each row and check for NaN in the first column
    for i in range(X.shape[0]):
        if not np.isnan(X[i, 0]):
            valid_indices.append(i)
    
    # Create new X and y with only valid rows
    X = X[valid_indices]
    y = y[valid_indices]
    coord = coord[valid_indices]

#     print('the size of dataset is:')
#     print(X.shape[0])

    if cutoff:
        if direction > 0: ### keep the eastern side
            masklon = coord[:, 1] > cutlon
        else:
            masklon = coord[:, 1] <= cutlon
    
        X = X[masklon]
        y = y[masklon]
        coord = coord[masklon]
    
#         print('the size of lon filtered dataset is:')
#         print(X.shape[0])
        #print(X)
    
    return X, y, coord

In [5]:
def model_train_test(X, y):
    # Initialize the RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Fit the model to the reshaped data
    model.fit(X, y)
    
    # After training, the model can be used for predictions or evaluation
    y_pred = model.predict(X)
    
    # Calculate accuracy by comparing the predictions with the true labels (y)
    accuracy = model.score(X, y)
    
    print(f"Training and testing using the whole dataset, accuracy is: {accuracy * 100:.2f}%")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    
    # Optionally, evaluate the model on the test set
    score = model.score(X_test, y_test)
    print(f"Testing using 20% of the whole dataset, accuracy is: {score*100:.2f}%")
    
    y_pred = model.predict(X_test)

    print("the number of elements for the prediction:")
    print(y_pred.shape[0])
    cm = confusion_matrix(y_test, y_pred, labels=['VZ', 'TZ', 'WZ'])
    
    # Plot the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VZ', 'TZ', 'WZ'])
    disp.plot(cmap='Blues', values_format='.0f')
    
    return accuracy

In [47]:
def model_across_years(X, y, X_test, y_test, coord_test, VZthreshold, WZthreshold):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    
    # After training, the model can be used for predictions or evaluation
    y_proba = model.predict_proba(X_test)
    y_pred = model.predict(X_test)
    y_change = y_pred.copy()
    
    #print("y_proba's shape:")
    #print(y_proba.shape)
    
    final_class_probs = np.max(y_proba, axis=1)

    vz_indices = (y_change == 'VZ') & (final_class_probs < VZthreshold)
    wz_indices = (y_change == 'WZ') & (final_class_probs < WZthreshold)

    # Update y_region_change to 'TZ' only for those indices
    y_change[vz_indices] = 'TZ'
    y_change[wz_indices] = 'TZ'
    
    y_change = np.array(y_change)
    y_test = np.array(y_test)

    # Calculate overall accuracy
    accuracy = accuracy_score(y_test, y_change)

    #print(f"Accuracy: {accuracy:.2f}")
    return y_change

In [7]:
def extract_coordinates(row):
    try:
        # Parse the JSON-like string to a dictionary
        data = ast.literal_eval(row)
        coordinates = data['coordinates']
        lon = round(coordinates[0], 10)
        lat = round(coordinates[1], 10)
        return coordinates[0], coordinates[1]  # lon, lat
    except (ValueError, KeyError):
        return None, None  # Handle parsing errors gracefully

In [8]:
def define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2):
    # Convert to pandas DataFrame for easier comparison
    # Convert to pandas DataFrame for easier comparison
    df1 = pd.DataFrame({'lat': coord_cal1[:,0], 'lon': coord_cal1[:,1], 'y': y_cal1})
    df2 = pd.DataFrame({'lat': coord_cal2[:,0], 'lon': coord_cal2[:,1], 'y': y_cal2})

    # Find common rows based on lat, lon, and y
    merged_df = df1.merge(df2, on=['lat', 'lon', 'y'], how='inner', indicator=True)

    ##print("Matching indices:", matching_indices)

    # Get indices from df1
    matching_indices_df1 = df1.index[df1[['lat', 'lon', 'y']].apply(tuple, axis=1).isin(
        merged_df[['lat', 'lon', 'y']].apply(tuple, axis=1)
    )].tolist()

    # Get indices from df2
    matching_indices_df2 = df2.index[df2[['lat', 'lon', 'y']].apply(tuple, axis=1).isin(
        merged_df[['lat', 'lon', 'y']].apply(tuple, axis=1)
    )].tolist()

    # Output results
    #print("Common elements and their indices in df1:")
    #print(df1.loc[matching_indices_df1])

    #print("\nIndices of the same elements in df2:")
    #print(df2.loc[matching_indices_df2])

    common_df1 = df1.loc[matching_indices_df1].to_numpy()
    common_df2 = df2.loc[matching_indices_df2].to_numpy()
    #print("common indices are:")
    #print(y_cal1[common_df1])
    #print('common indices are:')
    #print(y_cal1[matching_indices_df1])
    return common_df1, matching_indices_df1

In [9]:
def calculate_accuracy_target_year(common_df1, coord_pred, y_pred):
    ################## Evaluate the prediction accuracy for the target year #####################
    # Step 1: Find matching coordinates between coord_pred and common_df1
    common_coords = common_df1[:, :2]  # Extract lat, lon from common_df1

    # Find indices where coord_pred exists in common_coords
    matching_indices = [i for i, coord in enumerate(coord_pred) if any(np.all(coord == common_coords[j]) for j in range(common_coords.shape[0]))]

    # Step 2: Get y values from common_df1 based on matching indices
    y_true = np.array([common_df1[np.where((common_coords == coord_pred[i]).all(axis=1))[0][0], 2] for i in matching_indices])

    # Step 3: Get subset of y_pred for matching coordinates
    y_pred_subset = y_pred[matching_indices]

    # Step 4: Compute overall accuracy (Mean Absolute Error or % Accuracy)
    accuracy = np.mean(y_pred_subset == y_true) * 100 

    # Output results

    print(f"Accuracy of the predicted values compared to the consistency points between two cal years: {accuracy:.1f}%")
    return np.round(accuracy,1)

In [43]:
path = '/Volumes/Malonelab/Research/Whitezone/Data/KML/'
ifcombine = 1
sensorshift1 = 0

calyear1 = 2018
calyear2 = 2023

threshold = 0.65 ## using this to control the reclassification to TZ
folder = 'L8/'

#X_cal1, y_cal1, coord_cal1 = data_prep('Points/KML/' + str(calyear1) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal1, y_cal1, coord_cal1 = data_prep(path + str(calyear1) + '/', 'median.csv', 0, 0, 0, sensorshift1)
X_cal1 = X_cal1.astype(float)

#X_cal2, y_cal2, coord_cal2 = data_prep('Points/KML/' + str(calyear2) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb

X_cal2, y_cal2, coord_cal2 = data_prep(path + str(calyear2) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal2 = X_cal2.astype(float)

calyear3 = 2013
sensorshift2 = 1
#X_cal3, y_cal3, coord_cal3 = data_prep('Points/KML/' + str(calyear3) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3, y_cal3, coord_cal3 = data_prep(path + str(calyear3) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3 = X_cal3.astype(float)

In [ ]:
bands = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]

In [50]:
# -----------------------------
# Read CSV
# -----------------------------
csv_file = "/Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median.csv"

df = pd.read_csv(csv_file)

df[bands] = df[bands] * 0.0000275 - 0.2 ### convert it the float scale
# -----------------------------
# Original Landsat 7 ETM+ bands
# (scaled SR values)
# -----------------------------
B1 = df["SR_B1"]  # Blue
B2 = df["SR_B2"]  # Green
B3 = df["SR_B3"]  # Red
B4 = df["SR_B4"]  # NIR
B5 = df["SR_B5"]  # SWIR1
B7 = df["SR_B7"]  # SWIR2

# -----------------------------
# ETM+ -> OLI harmonization
# coefficients scaled to match
# integer SR values
# -----------------------------

df["SR_B1"] = 0.0003 + 0.8474 * B1    # Blue
df["SR_B3"] = 0.0088 + 0.8483 * B2    # Green
df["SR_B4"] = 0.0061 + 0.9047 * B3    # Red
df["SR_B5"] = 0.0412 + 0.8462 * B4    # NIR
df["SR_B6"] = 0.0254 + 0.8937 * B5    # SWIR1
df["SR_B7"] = 0.0172 + 0.9071 * B7    # SWIR2

df[bands] = (df[bands] + 0.2) / 0.0000275 ##convert back to original scale

# -----------------------------
# Recalculate vegetation indices
# -----------------------------

# NDVI
df["ndvi"] = (
    (df["SR_B5"] - df["SR_B4"])
    / (df["SR_B5"] + df["SR_B4"])
)

# NDMI
df["ndmi"] = (
    (df["SR_B5"] - df["SR_B6"])
    / (df["SR_B5"] + df["SR_B6"])
)

# SAVI
L = 0.5

df["savi"] = (
    ((df["SR_B5"] - df["SR_B4"])
     / (df["SR_B5"] + df["SR_B4"] + L))
    * (1 + L)
)

# -----------------------------
# Save
# -----------------------------
out = "/Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median_Roy.csv"

df.to_csv(out, index=False)

print("Saved:", out)
# Quick check
print(df[["SR_B2","SR_B3","SR_B4","SR_B5","SR_B6","SR_B7",
          "ndvi","ndmi","savi"]].head())

Saved: /Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median_Roy.csv
     SR_B2         SR_B3         SR_B4         SR_B5     SR_B6         SR_B7  \
0   9799.0   9735.764427  10090.376491  13500.551673  0.248765  12776.963025   
1  10018.0   9921.542127  10377.166391  14563.378873  0.250726  13020.822909   
2   9821.0   9754.427027  10381.689891  14309.997915  0.205923  11544.064109   
3  10106.0   9996.192527  10401.593291  14192.743273  0.165101  10999.804109   
4  10225.0  10097.140227  10452.789287  14672.115573  0.257362  13405.886859   

       ndvi      ndmi      savi  
0  0.144555  0.999963  0.216827  
1  0.167848  0.999966  0.251766  
2  0.159094  0.999971  0.238637  
3  0.154147  0.999977  0.231216  
4  0.167934  0.999965  0.251896  


In [51]:

csv_file = "/Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median.csv"

df = pd.read_csv(csv_file)

# -----------------------------
# Convert Landsat C2 SR DN
# to surface reflectance
# -----------------------------

df[bands] = df[bands] * 0.0000275 - 0.2

# -----------------------------
# Save original ETM+ bands
# -----------------------------

B1 = df["SR_B1"]  # Blue
B2 = df["SR_B2"]  # Green
B3 = df["SR_B3"]  # Red
B4 = df["SR_B4"]  # NIR
B5 = df["SR_B5"]  # SWIR1
B7 = df["SR_B7"]  # SWIR2

# -----------------------------
# ETM+ -> OLI harmonization
# -----------------------------

df["SR_B1"] = 0.9785 * B1 - 0.0095
df["SR_B3"] = 0.9542 * B2 - 0.0016
df["SR_B4"] = 0.9825 * B3 - 0.0022
df["SR_B5"] = 1.0073 * B4 - 0.0021
df["SR_B6"] = 1.0171 * B5 - 0.0030
df["SR_B7"] = 0.9949 * B7 + 0.0029

# -----------------------------
# Recalculate indices
# -----------------------------
df[bands] = (df[bands] + 0.2) / 0.0000275 ##convert back to original scale

df["NDVI"] = (
    (df["SR_B5"] - df["SR_B4"])
    / (df["SR_B5"] + df["SR_B4"])
)

df["NDMI"] = (
    (df["SR_B5"] - df["SR_B6"])
    / (df["SR_B5"] + df["SR_B6"])
)

L = 0.5
df["SAVI"] = (
    ((df["SR_B5"] - df["SR_B4"])
     / (df["SR_B5"] + df["SR_B4"] + L))
    * (1 + L)
)

# -----------------------------
# Save
# -----------------------------
out = "/Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median_Roy_RMA.csv"

df.to_csv(out, index=False)

print("Saved:", out)

Saved: /Volumes/Malonelab/Research/Whitezone/Data/KML/2013/median_Roy_RMA.csv


In [42]:
calyear3 = 2013
sensorshift2 = 1

X_cal3_Roy, y_cal3, coord_cal3 = data_prep(path + str(calyear3) + '/', 'median_Roy.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3_Roy = X_cal3_Roy.astype(float)

X_cal3_RMA, y_cal3, coord_cal3 = data_prep(path + str(calyear3) + '/', 'median_Roy_RMA.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3_RMA = X_cal3_RMA.astype(float)

In [13]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [10]:
################### Period 2013 - 2024 ###########################################

In [17]:
################ Calculating overall accuracy using consistency points (488 pts) ################
geyears = [2003, 2009, 2013, 2018, 2023]
vzlist = np.arange(0.5, 0.85, 0.05)

# Second array: 0.2 to 0.5 with 0.05 interval
wzthd = 0
for vzthd in vzlist:
    all_y_pred = []
    all_y_csd = []
    for predyear in range(2014, 2025):  # Loop from 2014 to 2018
    #     print('*************************************************')
    #     print('the year is')
    #     print(predyear)
        if predyear in geyears:
            continue

        inclusive = 1 if calyear1 <= predyear <= calyear2 else 0
        sensorshift = 0 if predyear >= 2014 else 1

        #X_pred, y_pred_fake, coord_pred = data_prep('Points/KML/Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
        X_pred, y_pred_fake, coord_pred = data_prep(path + 'Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
        X_pred = X_pred.astype(float)
        rows_with_nan = np.any(np.isnan(X_pred), axis=1)
        indices_with_nan = np.where(rows_with_nan)[0]
    #     print("indices with nan are:")
    #     print(indices_with_nan)

        ###################### Define consistency dataset ###################
        if 2013 < predyear < 2018:
            df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal3, y_cal3) ## this is to make the training data
        elif predyear > 2018:
            df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) 
        
        y_csd = y_cal1[common_indices]
        coord_csd = coord_cal1[common_indices] ## find the common y and coord for the two GE years

        ##### extracting unique values of coord_csd ##############
        coord_csd = np.array(coord_csd, dtype=float) 
        unique_coord_csd, unique_indices = np.unique(coord_csd, axis=0, return_index=True)

        # Extract subset of y_csd based on unique indices
        y_csd = y_csd[unique_indices]
        coord_csd = coord_csd[unique_indices]
        
        # Extract subset of coord_csd based on unique indices
        coord_pred = np.array(coord_pred, dtype=float) 
        unique_coord_pred, unique_indices = np.unique(coord_pred, axis=0, return_index=True)

        X_pred = X_pred[unique_indices]
        coord_pred = coord_pred[unique_indices]
        
        common_mask = (coord_csd[:, None] == coord_pred).all(-1).any(-1)
        indices_csd = np.where(common_mask)[0]  # Indices in coord_csd

        common_mask_pred = (coord_pred[:, None] == coord_csd).all(-1).any(-1)
        indices_pred = np.where(common_mask_pred)[0]  # Indices in coord_pred

        y_csd = y_csd[indices_csd] ## using the label from the common GE years
        X_csd = X_pred[indices_pred] ## using the X for the target year
        
        ############ Temporal transfer learning and accuracy assessment ######
        ############ Using the consistency points ! ##########################
        if ifcombine == 1:
            X_cal = np.vstack((X_cal1, X_cal2))
            y_cal = np.vstack((y_cal1.reshape(-1, 1), y_cal2.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)
        else: ## if it is 0, then only use the first one. 
            X_cal = np.vstack((X_cal1))
            y_cal = np.vstack((y_cal1.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)

        all_y_pred.append(y_pred)
        all_y_csd.append(y_csd)

    # Convert lists to NumPy arrays
    all_y_pred = np.concatenate(all_y_pred, axis=0)
    all_y_csd = np.concatenate(all_y_csd, axis=0)

    # Compute overall accuracy
    accuracy = accuracy_score(all_y_csd, all_y_pred)
    print("#######################################")
    print(f"vz thd: {vzthd:.2f}")
    print(f"Aggregating Accuracy: {accuracy:.4f}")


y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(899, 3)
#######################################
vz thd: 0.50
Aggregating Accuracy: 0.8476
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(899, 3)
#######################################
vz thd: 0.55
Aggregating Accuracy: 0.8485
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1120, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(1217, 3)
y_proba's shape:
(899, 3)
#######################################
vz thd: 0.60
Aggregating Accuracy: 0.8466
y_proba's shape:
(1120, 3)
y

In [53]:
################ Testing Roy's coefficients! ################
geyears = [2013, 2018]

wzthd = 0
vzthd = 0

all_y_pred = []
all_y_csd = []
for predyear in range(2013, 2018):  # Loop from 2014 to 2018
#     print('*************************************************')
#     print('the year is')
#     print(predyear)
    if predyear in geyears:
        continue

    inclusive = 1 if calyear1 <= predyear <= calyear2 else 0
    sensorshift = 0 if predyear >= 2014 else 1

    #X_pred, y_pred_fake, coord_pred = data_prep('Points/KML/Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
    X_pred, y_pred_fake, coord_pred = data_prep(path + 'Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
    X_pred = X_pred.astype(float)
    rows_with_nan = np.any(np.isnan(X_pred), axis=1)
    indices_with_nan = np.where(rows_with_nan)[0]
#     print("indices with nan are:")
#     print(indices_with_nan)

    ###################### Define consistency dataset ###################
    if 2013 < predyear < 2018:
        df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal3, y_cal3) ## this is to make the training data
    elif predyear > 2018:
        df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) 

    y_csd = y_cal1[common_indices]
    coord_csd = coord_cal1[common_indices] ## find the common y and coord for the two GE years

    ##### extracting unique values of coord_csd ##############
    coord_csd = np.array(coord_csd, dtype=float) 
    unique_coord_csd, unique_indices = np.unique(coord_csd, axis=0, return_index=True)

    # Extract subset of y_csd based on unique indices
    y_csd = y_csd[unique_indices]
    coord_csd = coord_csd[unique_indices]

    # Extract subset of coord_csd based on unique indices
    coord_pred = np.array(coord_pred, dtype=float) 
    unique_coord_pred, unique_indices = np.unique(coord_pred, axis=0, return_index=True)

    X_pred = X_pred[unique_indices]
    coord_pred = coord_pred[unique_indices]

    common_mask = (coord_csd[:, None] == coord_pred).all(-1).any(-1)
    indices_csd = np.where(common_mask)[0]  # Indices in coord_csd

    common_mask_pred = (coord_pred[:, None] == coord_csd).all(-1).any(-1)
    indices_pred = np.where(common_mask_pred)[0]  # Indices in coord_pred

    y_csd = y_csd[indices_csd] ## using the label from the common GE years
    X_csd = X_pred[indices_pred] ## using the X for the target year

    ############ Temporal transfer learning and accuracy assessment ######
    ############ Using the consistency points ! ##########################
    
    X_cal = np.vstack((X_cal1, X_cal3_Roy)) ### change here from X_cal2 to X_cal3_Roy or X_cal3_RMA as Roy transformed values. 
    y_cal = np.vstack((y_cal1.reshape(-1, 1), y_cal3.reshape(-1, 1)))
    y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)
    
    accuracy = accuracy_score(y_csd, y_pred)
   
    print("#######################################")
    print(predyear)
    print(f"Aggregating Accuracy: {accuracy:.4f}")


#######################################
2014
Aggregating Accuracy: 0.8223
#######################################
2015
Aggregating Accuracy: 0.8438
#######################################
2016
Aggregating Accuracy: 0.8339
#######################################
2017
Aggregating Accuracy: 0.8330


In [14]:
########## 80% and 20 % test for GE years #####################################

X_train1, X_test1, y_train1, y_test1, train_indices1, test_indices1 = train_test_split(
    X_cal1, y_cal1, np.arange(len(y_cal1)), 
    test_size=0.4, stratify=y_cal1, random_state=42
)

X_train2, X_test2, y_train2, y_test2, train_indices2, test_indices2 = train_test_split(
    X_cal2, y_cal2, np.arange(len(y_cal2)), 
    test_size=0.4, stratify=y_cal2, random_state=42
)


for vzthd in vzlist:
    all_y_pred = []
    all_y_true = []
    y_pred1 = model_across_years(X_train1, y_train1, X_test1, y_test1, coord_csd, vzthd, wzthd)
    y_pred2 = model_across_years(X_train2, y_train2, X_test2, y_test2, coord_csd, vzthd, wzthd)
    all_y_pred.append(y_pred1)
    all_y_pred.append(y_pred2)
    all_y_true.append(y_test1)
    all_y_true.append(y_test2)

    all_y_pred = np.concatenate(all_y_pred, axis=0)
    all_y_true = np.concatenate(all_y_true, axis=0)

    # Compute overall accuracy
    accuracy = accuracy_score(all_y_true, all_y_pred)
    print("#######################################")
    print(f"vz thd: {vzthd:.2f}")
    print(f"Aggregating Accuracy: {accuracy:.4f}")


#######################################
vz thd: 0.50
Aggregating Accuracy: 0.8189
#######################################
vz thd: 0.55
Aggregating Accuracy: 0.8138
#######################################
vz thd: 0.60
Aggregating Accuracy: 0.8087
#######################################
vz thd: 0.65
Aggregating Accuracy: 0.8036
#######################################
vz thd: 0.70
Aggregating Accuracy: 0.8019
#######################################
vz thd: 0.75
Aggregating Accuracy: 0.7976
#######################################
vz thd: 0.80
Aggregating Accuracy: 0.7849


In [15]:
ifcombine = 1
sensorshift1 = 1

calyear1 = 2009
calyear2 = 2013

folder = 'L7/'

X_cal1, y_cal1, coord_cal1 = data_prep('Points/KML/' + str(calyear1) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal1 = X_cal1.astype(float)

X_cal2, y_cal2, coord_cal2 = data_prep('Points/KML/' + str(calyear2) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal2 = X_cal2.astype(float)

calyear3 = 2003
sensorshift2 = 1
X_cal3, y_cal3, coord_cal3 = data_prep('Points/KML/' + str(calyear3) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3 = X_cal3.astype(float)

In [16]:
################ Calculating overall accuracy using consistency points (488 pts) ################
geyears = [2003, 2009, 2013, 2018, 2023]
vzlist = np.arange(0.5, 0.85, 0.05)

# Second array: 0.2 to 0.5 with 0.05 interval
wzlist = np.arange(0.2, 0.55, 0.05)
wzthd = 0
for vzthd in vzlist:
    all_y_pred = []
    all_y_csd = []
    for predyear in range(2004, 2013):  # Loop from 2014 to 2018
        if predyear in geyears:
            continue

        inclusive = 1 if calyear1 <= predyear <= calyear2 else 0
        sensorshift = 0 if predyear >= 2014 else 1

        X_pred, y_pred_fake, coord_pred = data_prep('Points/KML/Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
        X_pred = X_pred.astype(float)
        rows_with_nan = np.any(np.isnan(X_pred), axis=1)
        indices_with_nan = np.where(rows_with_nan)[0]

        ###################### Define consistency dataset ###################
        if 2009 < predyear < 2013:
            df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) ## this is to make the training data
        elif predyear < 2009:
            df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal3, y_cal3) 
        y_csd = y_cal1[common_indices]
        coord_csd = coord_cal1[common_indices] ## find the common y and coord for the two GE years
        
        ##### extracting unique values of coord_csd ##############
        coord_csd = np.array(coord_csd, dtype=float) 
        unique_coord_csd, unique_indices = np.unique(coord_csd, axis=0, return_index=True)

        # Extract subset of y_csd based on unique indices
        y_csd = y_csd[unique_indices]
        coord_csd = coord_csd[unique_indices]
        
        # Extract subset of coord_csd based on unique indices
        coord_pred = np.array(coord_pred, dtype=float) 
        unique_coord_pred, unique_indices = np.unique(coord_pred, axis=0, return_index=True)

        X_pred = X_pred[unique_indices]
        coord_pred = coord_pred[unique_indices]
        
        common_mask = (coord_csd[:, None] == coord_pred).all(-1).any(-1)
        indices_csd = np.where(common_mask)[0]  # Indices in coord_csd

        common_mask_pred = (coord_pred[:, None] == coord_csd).all(-1).any(-1)
        indices_pred = np.where(common_mask_pred)[0]  # Indices in coord_pred

        y_csd = y_csd[indices_csd] ## using the label from the common GE years
        X_csd = X_pred[indices_pred] ## using the X for the target year
        
        ############ Temporal transfer learning and accuracy assessment ######
        ############ Using the consistency points ! ##########################
        if ifcombine == 1:
            X_cal = np.vstack((X_cal1, X_cal2))
            y_cal = np.vstack((y_cal1.reshape(-1, 1), y_cal2.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)
        else: ## if it is 0, then only use the first one. 
            X_cal = np.vstack((X_cal1))
            y_cal = np.vstack((y_cal1.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)

        all_y_pred.append(y_pred)
        all_y_csd.append(y_csd)

    # Convert lists to NumPy arrays
    all_y_pred = np.concatenate(all_y_pred, axis=0)
    all_y_csd = np.concatenate(all_y_csd, axis=0)

    # Compute overall accuracy
    accuracy = accuracy_score(all_y_csd, all_y_pred)
    print("#######################################")
    print(f"vz thd: {vzthd:.2f}")
    print(f"Aggregating Accuracy: {accuracy:.4f}")


#######################################
vz thd: 0.50
Aggregating Accuracy: 0.8325
#######################################
vz thd: 0.55
Aggregating Accuracy: 0.8335
#######################################
vz thd: 0.60
Aggregating Accuracy: 0.8333
#######################################
vz thd: 0.65
Aggregating Accuracy: 0.8295
#######################################
vz thd: 0.70
Aggregating Accuracy: 0.8235
#######################################
vz thd: 0.75
Aggregating Accuracy: 0.8149
#######################################
vz thd: 0.80
Aggregating Accuracy: 0.8048


In [17]:
##################### Check for L5 period ############################
ifcombine = 1
sensorshift1 = 1
sensorshift2 = 1

calyear1 = 1985
calyear2 = 1994
calyear3 = 1999
calyear4 = 2003

threshold = 0.50 ## using this to control the reclassification to TZ
folder = 'L5/'

X_cal1, y_cal1, coord_cal1 = data_prep('Points/KML/' + str(calyear1) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal1 = X_cal1.astype(float)

X_cal2, y_cal2, coord_cal2 = data_prep('Points/KML/' + str(calyear2) + '/', 'median.csv', 0, 0, 0, sensorshift1) ## mixing 2022 Mar and 2023 Feb
X_cal2 = X_cal2.astype(float)

X_cal3, y_cal3, coord_cal3 = data_prep('Points/KML/' + str(calyear3) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal3 = X_cal3.astype(float)

X_cal4, y_cal4, coord_cal4 = data_prep('Points/KML/' + str(calyear4) + '/', 'median.csv', 0, 0, 0, sensorshift2) ## mixing 2022 Mar and 2023 Feb
X_cal4 = X_cal4.astype(float)

In [18]:
################ Calculating overall accuracy using consistency points (488 pts) ################

vzlist = np.arange(0.5, 0.85, 0.05)

# Second array: 0.2 to 0.5 with 0.05 interval
wzthd = 0
for vzthd in vzlist:
    all_y_pred = []
    all_y_csd = []
    for predyear in range(1984, 2004):  # Loop from 2014 to 2018
    #     print('*************************************************')
    #     print('the year is')
    #     print(predyear)
        if predyear == 1985 or predyear == 1994 or predyear == 1999 or predyear == 2003:
            continue

        inclusive = 1 if calyear1 <= predyear <= calyear2 else 0
        sensorshift = 0 if predyear >= 2014 else 1

        X_pred, y_pred_fake, coord_pred = data_prep('Points/KML/Emulation/' + folder, 'points_' + str(predyear) + '.csv', 0, 0, 0, sensorshift) ## mixing 2022 Mar and 2023 Feb
        X_pred = X_pred.astype(float)
        rows_with_nan = np.any(np.isnan(X_pred), axis=1)
        indices_with_nan = np.where(rows_with_nan)[0]
    #     print("indices with nan are:")
    #     print(indices_with_nan)

        ###################### Define consistency dataset ###################
        if 1984 <= predyear < 1994:
            df1, common_indices = define_consistency_dataset(coord_cal1, y_cal1, coord_cal2, y_cal2) ## this is to make the training data
            y_csd = y_cal1[common_indices]
            coord_csd = coord_cal1[common_indices] ## find the common y and coord for the two GE years
        elif 1994 < predyear < 1999:
            df1, common_indices = define_consistency_dataset(coord_cal2, y_cal2, coord_cal3, y_cal3) 
            y_csd = y_cal2[common_indices]
            coord_csd = coord_cal2[common_indices] ## find the common y and coord for the two GE years
        elif 1999 < predyear <= 2003:
            df1, common_indices = define_consistency_dataset(coord_cal3, y_cal3, coord_cal4, y_cal4) 
            y_csd = y_cal3[common_indices]
            coord_csd = coord_cal3[common_indices] ## find the common y and coord for the two GE years
        
        ##### extracting unique values of coord_csd ##############
        coord_csd = np.array(coord_csd, dtype=float) 
        unique_coord_csd, unique_indices = np.unique(coord_csd, axis=0, return_index=True)

        # Extract subset of y_csd based on unique indices
        y_csd = y_csd[unique_indices]
        coord_csd = coord_csd[unique_indices]
        
        # Extract subset of coord_csd based on unique indices
        coord_pred = np.array(coord_pred, dtype=float) 
        unique_coord_pred, unique_indices = np.unique(coord_pred, axis=0, return_index=True)

        X_pred = X_pred[unique_indices]
        coord_pred = coord_pred[unique_indices]
        
        common_mask = (coord_csd[:, None] == coord_pred).all(-1).any(-1)
        indices_csd = np.where(common_mask)[0]  # Indices in coord_csd

        common_mask_pred = (coord_pred[:, None] == coord_csd).all(-1).any(-1)
        indices_pred = np.where(common_mask_pred)[0]  # Indices in coord_pred

        y_csd = y_csd[indices_csd] ## using the label from the common GE years
        X_csd = X_pred[indices_pred] ## using the X for the target year

        ############ Temporal transfer learning and accuracy assessment ######
        ############ Using the consistency points ! ##########################
        if ifcombine == 1:
            X_cal = np.vstack((X_cal1, X_cal2))
            y_cal = np.vstack((y_cal1.reshape(-1, 1), y_cal2.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)
        else: ## if it is 0, then only use the first one. 
            X_cal = np.vstack((X_cal1))
            y_cal = np.vstack((y_cal1.reshape(-1, 1)))
            y_pred = model_across_years(X_cal, y_cal, X_csd, y_csd, coord_csd, vzthd, wzthd)

        all_y_pred.append(y_pred)
        all_y_csd.append(y_csd)

    # Convert lists to NumPy arrays
    all_y_pred = np.concatenate(all_y_pred, axis=0)
    all_y_csd = np.concatenate(all_y_csd, axis=0)

    # Compute overall accuracy
    accuracy = accuracy_score(all_y_csd, all_y_pred)
    print("#######################################")
    print(f"vz thd: {vzthd:.2f}")
    print(f"Aggregating Accuracy: {accuracy:.4f}")

#######################################
vz thd: 0.50
wz thd: 0.00
Aggregating Accuracy: 0.7530
#######################################
vz thd: 0.55
wz thd: 0.00
Aggregating Accuracy: 0.7548
#######################################
vz thd: 0.60
wz thd: 0.00
Aggregating Accuracy: 0.7565
#######################################
vz thd: 0.65
wz thd: 0.00
Aggregating Accuracy: 0.7550
#######################################
vz thd: 0.70
wz thd: 0.00
Aggregating Accuracy: 0.7505
#######################################
vz thd: 0.75
wz thd: 0.00
Aggregating Accuracy: 0.7437
#######################################
vz thd: 0.80
wz thd: 0.00
Aggregating Accuracy: 0.7336
